# Tutorial: Milestone 0 Contract Freeze + Parity Harness

Audience:
- Engineers validating Phase 1 API contract parity during Python to Rust migration.

Prerequisites:
- Run this notebook from the repository root.
- `uv` and `cargo` are installed.

Learning goals:
- Verify canonical parity fixtures exist and load.
- Run the parity harness against Python backend.
- Regenerate fixtures in guarded mode.
- Confirm Rust daemon baseline tests are green.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
FIXTURE_PATH = REPO_ROOT / "tests/parity/fixtures/phase1/corpus.json"
assert (REPO_ROOT / "pyproject.toml").exists(), "Run notebook from repo root."
print(f"repo: {REPO_ROOT}")
print(f"fixture path: {FIXTURE_PATH}")


## Step 1 - Verify Python parity harness gate

Expected result: command exits 0 and reports `1 passed`.


In [ ]:
result = subprocess.run(
    ["uv", "run", "pytest", "tests/test_phase1_parity.py", "-q"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Parity gate failed."


## Step 2 - Inspect canonical fixture corpus

Expected result: fixture file exists and includes cases for all Phase 1 endpoints.


In [ ]:
payload = json.loads(FIXTURE_PATH.read_text(encoding="utf-8"))
cases = payload["cases"]
names = [item["name"] for item in cases]
print(f"fixture schema_version: {payload['schema_version']}")
print(f"total cases: {len(cases)}")
print("first 8 cases:", names[:8])
required_prefixes = [
    "dataset_open_",
    "session_create_",
    "view_create_",
    "view_get_",
    "view_update_",
    "export_viewstate_",
    "import_viewstate_",
    "render_image_",
]
for prefix in required_prefixes:
    assert any(name.startswith(prefix) for name in names), f"missing fixture group: {prefix}"


## Step 3 - Regenerate fixtures in guarded mode

Expected result: regeneration command exits 0 and does not change tracked fixtures when behavior is unchanged.


In [ ]:
regen_env = dict(os.environ)
regen_env["LUCIDA_REGEN_PARITY_FIXTURES"] = "1"
regen = subprocess.run(
    ["uv", "run", "pytest", "tests/test_phase1_parity.py", "-q"],
    cwd=REPO_ROOT,
    env=regen_env,
    capture_output=True,
    text=True,
)
print(regen.stdout)
if regen.stderr:
    print(regen.stderr)
assert regen.returncode == 0, "Fixture regeneration failed."
diff = subprocess.run(
    ["git", "diff", "--", str(FIXTURE_PATH)],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
assert diff.stdout.strip() == "", "Fixture corpus changed after regeneration. Inspect diff."
print("fixture corpus stable after regeneration")


## Step 4 - Validate Rust daemon baseline

Expected result: `cargo test` exits 0 for the new workspace/crate scaffold.


In [ ]:
cargo = subprocess.run(
    ["cargo", "test"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(cargo.stdout)
if cargo.stderr:
    print(cargo.stderr)
assert cargo.returncode == 0, "cargo test failed."


## Milestone 0 done

If all assertions passed, Milestone 0 gates are satisfied for:
- Python parity fixture corpus + harness validation.
- Guarded fixture regeneration workflow.
- Rust daemon scaffold baseline tests.
